### Hybrid Retriever - Combining Dense and Sparse Retriever

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever # for sparse
from langchain_classic.retrievers import EnsembleRetriever  # combine two retriever here dense and sparse
from langchain_core.documents import Document

In [5]:

# Step 1: Sample documents

docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]


# Step 2: Dense Retriever (FAISS + HuggingFace)

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs,embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5440.84it/s]


In [8]:
#Step 3
### Sparse Retriever  -BM25 <-- It is an algorithm use to covert the text into vector
spare_retriever = BM25Retriever.from_documents(docs)
spare_retriever.k=3 # top -k documents to retriever

# Step 4
# Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,spare_retriever],
    weights=[0.7,0.3]
)



In [9]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E0A73A1370>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001E0A8D9BFB0>, k=3)], weights=[0.7, 0.3])

In [10]:
# step 5 : Query and get result
query ="How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6 : print result 

for i, doc in enumerate(results):
    print(f"\n Document {i+1}:\n{doc.page_content}")


 Document 1:
LangChain helps build LLM applications.

 Document 2:
Langchain can be used to develop agentic ai application.

 Document 3:
Langchain has many types of retrievers.

 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()

# Check key
print(os.getenv("GROQ_API_KEY"))

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


In [14]:
# Step 5 
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question:
{input}
                                        """)

# Step 6 LLM model
### LLM
llm = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.2
)

llm

ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E0A73A1220>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E0AA37CA40>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [15]:
### Create stuff Document Chain
document_chain =create_stuff_documents_chain(llm=llm,prompt=prompt)
# create full rag chain 
rag_chain = create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E0A73A1370>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001E0A8D9BFB0>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion:\n{input}\n                                  

In [16]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")


✅ Answer:
 To build an app using LLMs (Large Language Models), you can use LangChain, which is a framework that helps build LLM applications. Here's a general outline of the steps you can follow:

1. **Choose a LangChain component**: LangChain has various components, including retrievers, which are used to fetch relevant information from a database or knowledge graph. You can choose a retriever that fits your needs, such as Pinecone, a vector database for semantic search.
2. **Define your application's requirements**: Determine what kind of app you want to build, such as a chatbot, a question-answering system, or a content generation tool.
3. **Select a suitable LLM**: Choose a pre-trained LLM that aligns with your application's requirements. LangChain supports various LLMs, including popular ones like LLaMA and BERT.
4. **Integrate the LLM with LangChain**: Use LangChain's APIs to integrate the LLM with your application. This will enable you to leverage the LLM's capabilities, such as